# Delivery Operations Analysis

Exploratory analysis of cost, volume, rider productivity, distance, and cost anomalies using the transformed SQLite model.

In [ ]:
from pathlib import Path
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='deep')
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DB_PATH = ROOT / 'delivery_operations.db'
connection = sqlite3.connect(DB_PATH)
deliveries = pd.read_sql_query('SELECT * FROM delivery_metrics', connection, parse_dates=['order_date'])
rider_daily = pd.read_sql_query('SELECT * FROM rider_daily_metrics', connection, parse_dates=['metric_date'])
deliveries.shape, rider_daily.shape

## Data Quality and KPI Snapshot

In [ ]:
summary = pd.Series({
    'Delivered orders': len(deliveries),
    'Distinct riders': deliveries['rider_id'].nunique(),
    'Average CPD': deliveries['delivery_cost'].mean(),
    'Average distance': deliveries['delivery_distance_km'].mean(),
    'Average productivity': rider_daily['productivity'].mean(),
    'Anomalies': deliveries['is_cost_anomaly'].sum(),
    'Profit margin %': deliveries['profit'].sum() / deliveries['revenue'].sum() * 100,
})
summary.to_frame('value').round(2)

In [ ]:
assert deliveries['order_id'].is_unique
assert deliveries[['delivery_cost', 'revenue', 'delivery_distance_km']].notna().all().all()
assert set(deliveries['employment_type']) == {'1P', '3P'}
deliveries.describe(include='all').T

## Monthly CPD Trend

In [ ]:
monthly = (deliveries.set_index('order_date')
           .resample('MS')
           .agg(orders=('order_id', 'count'), cost=('delivery_cost', 'sum')))
monthly['cpd'] = monthly['cost'] / monthly['orders']
fig, ax = plt.subplots(figsize=(12, 5))
sns.lineplot(data=monthly, x=monthly.index, y='cpd', marker='o', ax=ax)
ax.set(title='Monthly Cost Per Delivery', xlabel='', ylabel='CPD (INR)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Rider Productivity Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(data=rider_daily, x='productivity', hue='employment_type', bins=30, kde=True, ax=ax)
ax.set(title='Rider-Day Productivity Distribution', xlabel='Delivered Orders per Shift Hour')
plt.tight_layout()
plt.show()

## Delivery Distance Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(deliveries['delivery_distance_km'], bins=40, kde=True, color='#5BA8FF', ax=ax)
ax.axvline(deliveries['delivery_distance_km'].median(), color='#E96B78', linestyle='--', label='Median')
ax.set(title='Delivery Distance Distribution', xlabel='Distance (km)')
ax.legend()
plt.tight_layout()
plt.show()

## Orders by City

In [ ]:
city_orders = deliveries['city'].value_counts().rename_axis('city').reset_index(name='orders')
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=city_orders, x='orders', y='city', hue='city', legend=False, ax=ax)
ax.set(title='Delivered Orders by City', xlabel='Delivered Orders', ylabel='')
plt.tight_layout()
plt.show()

## Cost Anomalies

In [ ]:
mean_cost = deliveries['delivery_cost'].mean()
std_cost = deliveries['delivery_cost'].std(ddof=0)
threshold = mean_cost + 2 * std_cost
deliveries['computed_anomaly'] = deliveries['delivery_cost'].gt(threshold)
assert (deliveries['computed_anomaly'].astype(int) == deliveries['is_cost_anomaly']).all()

fig, ax = plt.subplots(figsize=(11, 6))
sample = deliveries.sample(min(12000, len(deliveries)), random_state=42)
sns.scatterplot(data=sample, x='delivery_distance_km', y='delivery_cost',
                hue='computed_anomaly', alpha=0.55, ax=ax)
ax.axhline(threshold, color='#E96B78', linestyle='--', label=f'Threshold: INR {threshold:.2f}')
ax.set(title='Distance vs Delivery Cost with Statistical Anomalies',
       xlabel='Distance (km)', ylabel='Delivery Cost (INR)')
plt.tight_layout()
plt.show()

## 1P vs 3P Performance

In [ ]:
fleet = (deliveries.groupby('employment_type')
         .agg(orders=('order_id', 'count'), avg_cpd=('delivery_cost', 'mean'),
              avg_distance=('delivery_distance_km', 'mean'), payout_variance=('payout_variance', 'sum')))
fleet['avg_productivity'] = rider_daily.groupby('employment_type')['productivity'].mean()
fleet.round(2)

## Rider Productivity Heatmap

In [ ]:
heat = rider_daily.assign(weekday=rider_daily['metric_date'].dt.day_name())
weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
heat = heat.pivot_table(index='city', columns='weekday', values='productivity', aggfunc='mean').reindex(columns=weekday_order)
fig, ax = plt.subplots(figsize=(11, 5))
sns.heatmap(heat, annot=True, fmt='.3f', cmap='YlGnBu', ax=ax)
ax.set(title='Average Rider Productivity by City and Weekday', xlabel='', ylabel='')
plt.tight_layout()
plt.show()

In [ ]:
connection.close()